In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data=pd.read_csv("C:/riya/csv/StudentPerformanceFactors.csv")
data

data summary

In [ ]:
data.info()

In [ ]:
data.describe()

data cleaning

checking duplicate rows

In [ ]:
# remove duplicates
data.duplicated()

In [ ]:
data.duplicated().sum()

In [ ]:
data=data.drop_duplicates()

null values


In [ ]:
data.isna()

In [ ]:
data.isna().sum()

filter the column

In [ ]:
data=data.dropna()

In [ ]:
data.shape

data visualization

1. Histogram – Exam Score Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=data, x='Motivation_Level',
              order=data['Motivation_Level'].value_counts().index)
plt.title('Distribution of Motivation Levels')
plt.show()

2. Box Plot – Exam Score by Gender

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=data, x='Gender', y='Exam_Score')
plt.title('Exam Score by Gender')
plt.show()

3. Violin Plot – Exam Score by School Type

In [ ]:
plt.figure(figsize=(8,5))
sns.violinplot(data=data, x='School_Type', y='Exam_Score')
plt.title('Exam Score Distribution by School Type')
plt.show()

4. Scatter Plot – Hours Studied vs Exam Score

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=data,
                x='Hours_Studied',
                y='Exam_Score',
                hue='Gender')
plt.title('Hours Studied vs Exam Score')
plt.show()

5. Bar Plot – Average Exam Score by Motivation Level

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=data,
            x='Motivation_Level',
            y='Exam_Score')
plt.title('Average Exam Score by Motivation Level')
plt.show()

6. Count Plot – Family Income Levels

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=data, x='Family_Income')
plt.title('Family Income Distribution')
plt.show()

7. Heatmap – Correlation Matrix

In [ ]:
plt.figure(figsize=(10,6))

corr = data[['Hours_Studied',
           'Attendance',
           'Sleep_Hours',
           'Previous_Scores',
           'Tutoring_Sessions',
           'Physical_Activity',
           'Exam_Score']].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

8. Line Plot – Average Exam Score by Hours Studied

In [ ]:
avg_score = data.groupby('Hours_Studied')['Exam_Score'].mean().reset_index()

plt.figure(figsize=(8,5))
sns.lineplot(data=avg_score,
             x='Hours_Studied',
             y='Exam_Score')
plt.title('Average Exam Score by Study Hours')
plt.show()

9. Pair Plot – Numerical Features

In [ ]:
sns.pairplot(
    data[['Hours_Studied',
        'Attendance',
        'Sleep_Hours',
        'Previous_Scores',
        'Exam_Score']]
)
plt.show()

10. Strip Plot – Exam Score by Parental Involvement

In [ ]:
plt.figure(figsize=(8,5))
sns.stripplot(data=data,
              x='Parental_Involvement',
              y='Exam_Score',
              jitter=True)
plt.title('Exam Score by Parental Involvement')
plt.show()

feature engineering

 1. Handle missing values

In [ ]:
data.fillna(data.mode().iloc[0], inplace=True)

 2. Create 1 feature (optional)

In [ ]:
data["Study_Efficiency"] = (
    data["Hours_Studied"] * data["Attendance"] / 100
)


 3. Encode categorical variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

categorical_cols = data.select_dtypes(include="object").columns

for col in categorical_cols:
    data[col] = le.fit_transform(data[col])

4.Correlation with Target (Simple)

In [ ]:
# Correlation with Exam_Score
corr = data.corr(numeric_only=True)["Exam_Score"].sort_values(ascending=False)

print(corr)

5.SelectKBest (Recommended)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

X = data.drop("Exam_Score", axis=1)
y = data["Exam_Score"]

selector = SelectKBest(score_func=f_regression, k=10)
X_new = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("Selected Features:")
print(selected_features)

feature selection

Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_regression

mi = mutual_info_regression(X, y)

mi_scores = pd.Series(mi, index=X.columns).sort_values(ascending=False)
print(mi_scores)

Random Forest Importance

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

Variance Threshold

In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.1)
selector.fit(X)

print(X.columns[selector.get_support()])

Recursive Feature Elimination (RFE)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

model = LinearRegression()

rfe = RFE(model, n_features_to_select=10)
rfe.fit(X, y)

print(X.columns[rfe.support_])

Chi-Square (for categorical features)

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(chi2, k=10)
selector.fit(abs(X), y)

print(X.columns[selector.get_support()])

SelectFromModel (Tree-based)

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X, y)

selector = SelectFromModel(model, threshold="mean")
selector.fit(X, y)

print(X.columns[selector.get_support()])

Top Features by Correlation Threshold

In [ ]:
corr = data.corr(numeric_only=True)["Exam_Score"]

selected = corr[abs(corr) > 0.2].index
selected = selected.drop("Exam_Score")

print(selected)

Top Features by Correlation Threshold

In [ ]:
corr = data.corr(numeric_only=True)["Exam_Score"]

selected = corr[abs(corr) > 0.2].index
selected = selected.drop("Exam_Score")

print(selected)

data split/partion

1. Basic 80–20 Split

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop("Exam_Score", axis=1)
y = data["Exam_Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

data transformation

Feature scaling (VERY IMPORTANT)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

SVC model initialization

In [ ]:
from sklearn.svm import SVC

model = SVC()

Train model

In [ ]:
model.fit(X_train, y_train)

Prediction

In [ ]:
y_pred = model.predict(X_test)

Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

model visualization

Actual vs Predicted Plot (Most Important)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual Exam Score")
plt.ylabel("Predicted Exam Score")
plt.title("Actual vs Predicted Scores")
plt.show()

model deployment

Save your model

In [ ]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))  # only if you used scaling

model exporting

Export Model using Pickle (Most Common)

In [ ]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))

scaler

In [ ]:
pickle.dump(scaler, open("scaler.pkl", "wb"))